# Dự đoán khách hàng có chốt sale hay không

Các bước: đọc dữ liệu → tạo nhãn → chọn cột → chia 80/15/5 → huấn luyện → kiểm tra.

- **Train 80%**: mô hình học
- **Test 15%**: kiểm tra trong lúc làm
- **Holdout 5%**: giữ lại, mô hình chưa từng thấy, dùng cuối cùng để so dự báo với thực tế

## Bước 1: Đọc dữ liệu và tạo nhãn

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

data = pd.read_excel("Courses_of_Interest_Processed.xlsx")

# Nhãn: có ngày chốt sale = 1 (chốt), để trống = 0 (không chốt)
data["chot"] = data["thoi_gian_chuyen_sang_chot_sale"].notna().astype(int)

print("Số dòng:", len(data))
print(data["chot"].value_counts())

## Bước 2: Tạo thêm cột từ ngày tháng

In [ ]:
ngay_dang_ky = pd.to_datetime(data["ngay_tao"])
ngay_tu_van = pd.to_datetime(data["thoi_gian_chuyen_sang_tu_van"])

data["gio_dang_ky"] = ngay_dang_ky.dt.hour
data["thu_dang_ky"] = ngay_dang_ky.dt.dayofweek       # 0 = thứ 2, 6 = chủ nhật
data["thang_dang_ky"] = ngay_dang_ky.dt.month
data["co_tu_van"] = ngay_tu_van.notna().astype(int)   # 1 = đã chuyển sang tư vấn
data["so_ngay_den_tu_van"] = (ngay_tu_van - ngay_dang_ky).dt.days
data.loc[data["so_ngay_den_tu_van"] < 0, "so_ngay_den_tu_van"] = None   # ngày âm là lỗi

## Bước 3: Chọn cột đầu vào

- Cột chữ (nhóm): đổi thành 0/1 bằng `get_dummies`
- Cột số bị thiếu: điền -1
- Không dùng ngày chốt sale (vì đó chính là đáp án)

In [ ]:
cot_chu = ["MKT_xu_ly_data", "nha_mang", "quan_tam_khoa_hoc", "sentimatic", "tu_van_vien"]
cot_so = ["gio_dang_ky", "thu_dang_ky", "thang_dang_ky", "co_tu_van", "so_ngay_den_tu_van"]

X = data[cot_chu + cot_so].copy()
X[cot_chu] = X[cot_chu].fillna("Chua ro")
X[cot_so] = X[cot_so].fillna(-1)
X = pd.get_dummies(X, columns=cot_chu)

y = data["chot"]
print("Số cột đầu vào:", X.shape[1])

## Bước 4: Chia dữ liệu 80 / 15 / 5

Tách 5% holdout trước, rồi chia phần còn lại thành 80% train và 15% test. `stratify=y` giữ tỷ lệ chốt giống nhau ở cả 3 tập.

In [ ]:
X_con_lai, X_holdout, y_con_lai, y_holdout = train_test_split(X, y, test_size=0.05, stratify=y, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_con_lai, y_con_lai, test_size=15/95, stratify=y_con_lai, random_state=42)

print("Train  :", len(X_train), "dòng, tỷ lệ chốt", round(y_train.mean(), 3))
print("Test   :", len(X_test), "dòng, tỷ lệ chốt", round(y_test.mean(), 3))
print("Holdout:", len(X_holdout), "dòng, tỷ lệ chốt", round(y_holdout.mean(), 3))

## Bước 5: Huấn luyện Random Forest (chỉ trên train)

In [ ]:
model = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

## Bước 6: Kiểm tra trên train, test và holdout

In [ ]:
def danh_gia(X_, y_):
    du_bao = model.predict(X_)                      # 1 = chốt, 0 = không chốt
    xac_suat = model.predict_proba(X_)[:, 1]        # xác suất chốt
    return pd.Series({
        "Accuracy (đúng bao nhiêu %)": accuracy_score(y_, du_bao),
        "Precision (báo chốt thì đúng bao nhiêu %)": precision_score(y_, du_bao),
        "Recall (bắt được bao nhiêu % khách chốt thật)": recall_score(y_, du_bao),
        "F1": f1_score(y_, du_bao),
        "ROC-AUC": roc_auc_score(y_, xac_suat),
    })

bang = pd.DataFrame({
    "Train 80%": danh_gia(X_train, y_train),
    "Test 15%": danh_gia(X_test, y_test),
    "Holdout 5%": danh_gia(X_holdout, y_holdout),
})
bang.round(3)

Nếu cột **Holdout** gần giống cột **Test** thì mô hình dự báo ổn trên dữ liệu chưa từng thấy.

## Bước 7: Đối chiếu từng khách trong 5% holdout

In [ ]:
ket_qua = data.loc[X_holdout.index, ["STT", "khach_hang", "tu_van_vien"]].copy()
ket_qua["thuc_te"] = y_holdout.map({1: "Chốt", 0: "Không chốt"})
ket_qua["du_bao"] = pd.Series(model.predict(X_holdout), index=X_holdout.index).map({1: "Chốt", 0: "Không chốt"})
ket_qua["xac_suat_chot"] = model.predict_proba(X_holdout)[:, 1].round(3)
ket_qua["dung_sai"] = (ket_qua["thuc_te"] == ket_qua["du_bao"]).map({True: "Đúng", False: "Sai"})

print("Dự báo đúng:", (ket_qua["dung_sai"] == "Đúng").sum(), "/", len(ket_qua))
pd.crosstab(ket_qua["thuc_te"], ket_qua["du_bao"], margins=True)   # hàng = thực tế, cột = dự báo

In [ ]:
ket_qua.sort_values("xac_suat_chot", ascending=False).head(10)

In [ ]:
ket_qua.to_excel("Holdout_5pct_DuBao_vs_ThucTe.xlsx", index=False)

## Bước 8: Cột nào quan trọng nhất?

In [ ]:
quan_trong = pd.Series(model.feature_importances_, index=X.columns)

# gộp các cột 0/1 của cùng một biến gốc lại
def ten_goc(ten_cot):
    for c in cot_chu:
        if ten_cot.startswith(c + "_"):
            return c
    return ten_cot

quan_trong.groupby(ten_goc).sum().sort_values(ascending=False).round(3)

In [ ]:
import matplotlib.pyplot as plt

top = quan_trong.groupby(ten_goc).sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 5))
plt.barh(top.index[::-1], top.values[::-1])      # đảo ngược để biến quan trọng nhất nằm trên cùng
plt.title("Biến quan trọng nhất trong dự báo chốt sale")
plt.xlabel("Mức quan trọng")
plt.tight_layout()
plt.show()